<a href="https://colab.research.google.com/github/Tech-Matt/tiny-transformers/blob/main/transformer_from_scratch_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Input Embedding

In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
class InputEmbeddings(nn.Module):
    """
    Input Embeddings — translates token IDs (plain integers) into rich
    vectors of numbers that carry semantic meaning.

    Why we need it: the transformer cannot work with raw words or even
    raw integers. It needs every token to be represented as a vector of
    numbers (an embedding) so that mathematical operations like attention
    can find relationships between words.

    How it works: internally nn.Embedding is just a lookup table — a big
    matrix of shape (vocab_size, d_model). Each row is one word's vector.
    Given a token ID, it simply returns that row. The values start random
    but are gradually learned during training so that words with similar
    meanings end up with similar vectors.

    Example with d_model=4:
        token ID 0 "the"  →  [ 0.21, -0.54,  0.87,  0.13]
        token ID 1 "cat"  →  [ 0.92, -0.11,  0.34,  0.67]
        token ID 2 "sat"  →  [-0.45,  0.78,  0.02, -0.33]
    """

    def __init__(self, d_model: int, vocab_size: int):
        super().__init__()

        # d_model: how many numbers represent each word (e.g. 512).
        # A larger d_model means more capacity to encode meaning,
        # but also more computation. We store it because we need it
        # in forward() for the scaling step.
        self.d_model = d_model

        # vocab_size: how many unique tokens exist in our vocabulary.
        # This determines the number of rows in the lookup table —
        # one row per possible token.
        self.vocab_size = vocab_size

        # nn.Embedding creates the lookup table of shape (vocab_size, d_m

## 2. Positional Encoding

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Positional Encoding — injects information about token position into
    the word embeddings.

    Why we need it: the transformer processes all tokens simultaneously
    (in parallel), so unlike an RNN it has no built-in sense of order.
    "The cat sat" and "sat the cat" would look identical without this.

    How it works: we compute a fixed vector of sin/cos values for each
    position, then add it on top of the word embedding. The result is a
    single vector that carries both WHAT the word is and WHERE it sits.

    The sin/cos values use many different frequencies — like the hands of
    a clock at different speeds — so every position gets a unique pattern
    of values that the model can learn to read.
    """

    def __init__(self, d_model: int, seq_len: int, dropout: float):
        super().__init__()

        # d_model: how many dimensions each embedding vector has (e.g. 512).
        # We need this to know how many sin/cos values to compute per position.
        self.d_model = d_model

        # seq_len: the maximum number of tokens we will ever process.
        # We pre-compute positional encodings for ALL positions up to this
        # limit once at startup, then just slice out what we need at runtime.
        self.seq_len = seq_len

        # Dropout randomly zeroes some values during training with probability
        # `dropout` (e.g. 0.1 means 10% of values become 0).
        # This prevents the model from relying too heavily on any single value,
        # which helps it generalize better to sentences it has never seen.
        self.dropout = nn.Dropout(dropout)

        # --- Build the full positional encoding table (done once, at startup) ---

        # Create an empty table of shape (seq_len, d_model).
        # Rows = positions (0, 1, 2, ... seq_len-1)
        # Columns = embedding dimensions (0, 1, 2, ... d_model-1)
        # We will fill in each cell with the appropriate sin or cos value below.
        pe = torch.zeros(seq_len, d_model)

        # Create a column vector of position indices: [[0], [1], [2], ..., [seq_len-1]]
        # torch.arange gives a flat list [0, 1, 2, ...], shape (seq_len,)
        # .unsqueeze(1) inserts a new dimension making it shape (seq_len, 1) —
        # a column vector. This shape is needed so PyTorch can broadcast it
        # against div_term (which is a row) to fill the whole table in one step.
        position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)

        # Compute the frequencies for each dimension pair.
        # The original formula is:  1 / 10000^(2i / d_model)
        # We rewrite it as:         exp(2i * -log(10000) / d_model)
        # These are mathematically identical — the exp/log form is preferred
        # because computing large powers like 10000^(512/512) can cause
        # floating point errors, while exp() is numerically well-behaved.
        # torch.arange(0, d_model, 2) gives [0, 2, 4, ...] — the even indices,
        # one frequency value per sin/cos pair.
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        # Fill the even columns (0, 2, 4, ...) with sin values.
        # position * div_term broadcasts (seq_len, 1) × (d_model/2,)
        # into a (seq_len, d_model/2) grid — every position × every frequency.
        # pe[:, 0::2] means: all rows, every other column starting from 0.
        pe[:, 0::2] = torch.sin(position * div_term)

        # Fill the odd columns (1, 3, 5, ...) with cos values.
        # Same frequencies as sin — cos gives the "second reading" at each
        # frequency so that two positions that share a sin value can still
        # be told apart by their cos value.
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add a batch dimension at position 0: (seq_len, d_model) → (1, seq_len, d_model)
        # PyTorch processes data in batches of shape (batch_size, seq_len, d_model).
        # The 1 here means "one copy of pe" — PyTorch will automatically broadcast
        # (stretch) it to match however many sentences are in the batch, without
        # physically copying the data in memory.
        pe = pe.unsqueeze(0)

        # Register pe as a "buffer" rather than a plain attribute or a Parameter.
        # A buffer is a tensor that:
        #   - is NOT learned (no gradient, never updated by backpropagation)
        #   - IS saved when you call torch.save(model)
        #   - IS moved to GPU automatically when you call model.cuda()
        # This is the right choice for pe because it is fixed math, not a
        # learned weight — but we still want it to travel with the model.
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: (batch_size, actual_seq_len, d_model)
        # pe shape: (1, seq_len, d_model)  — precomputed for the maximum length

        # Slice pe down to the actual length of this input.
        # pe was built for the worst-case maximum length (e.g. 512 tokens),
        # but most sentences are shorter. x.shape[1] is the real length,
        # so pe[:, :x.shape[1], :] means:
        #   dim 0 →  :           keep the single batch copy (broadcasts to full batch)
        #   dim 1 →  :x.shape[1] keep only the first N rows (N = actual sentence length)
        #   dim 2 →  :           keep all d_model dimensions
        # .requires_grad_(False) explicitly tells PyTorch: do not track gradients
        # through pe during backpropagation — it is fixed, nothing to learn here.
        x = x + (self.pe[:, :x.shape[1], :]).requires_grad_(False)

        # Apply dropout to the combined (word + position) embedding.
        # Randomly zeroes some values during training to improve generalization.
        # At inference time (model.eval()) dropout is automatically disabled.
        return self.dropout(x)

## 3. Layer Normalization

In [ ]:
class LayerNormalization(nn.Module):
    """
    Layer Normalization — stabilizes training by rescaling each token's
    embedding to have mean 0 and std 1, then applying a learned rescaling.

    Why we need it: without normalization, values across the network can
    grow very large or very small, making training unstable and slow.
    This layer ensures values stay in a consistent range at every step.
    """

    def __init__(self, eps: float = 10**-6) -> None:
        super().__init__()

        # eps (epsilon): a tiny number added to the std before dividing.
        # This prevents division by zero in the rare case where all values
        # in a token's embedding are identical (std would be 0).
        # 10**-6 = 0.000001 — small enough to be harmless otherwise.
        self.eps = eps

        # alpha: a learnable scale factor, initialized to 1 (no scaling).
        # After normalizing, the model may need to re-scale the values —
        # alpha lets it learn how much. nn.Parameter tells PyTorch to
        # update this value during backpropagation, just like a weight.
        self.alpha = nn.Parameter(torch.ones(1))

        # bias: a learnable shift, initialized to 0 (no shift).
        # After normalizing, the model may need to shift the values up or
        # down — bias lets it learn by how much.
        # Together, alpha and bias let the model "undo" normalization if
        # needed, giving it full flexibility.
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        # x shape: (batch_size, seq_len, d_model)
        # Each token (one row of d_model values) is normalized independently.

        # Compute the mean of each token's embedding values.
        # dim=-1 means "average across the last dimension" (d_model),
        # so each token gets its own mean — not averaged across tokens.
        # keepdim=True keeps the shape as (batch, seq_len, 1) instead of
        # (batch, seq_len), so it can broadcast back against x correctly.
        mean = x.mean(dim=-1, keepdim=True)

        # Compute the standard deviation of each token's embedding values.
        # Same logic as mean: per-token, last dimension, shape kept.
        # std tells us how spread out the values are around the mean.
        std = x.std(dim=-1, keepdim=True)

        # Normalize: subtract mean so values are centered at 0,
        # then divide by (std + eps) so values are scaled to std of 1.
        # eps is added to std to avoid dividing by zero.
        # Then apply the learned rescaling: alpha stretches/shrinks the
        # values and bias shifts them up or down.
        # If the model learns alpha=1 and bias=0 it effectively does
        # nothing, preserving the normalized values as-is.
        return self.alpha * (x - mean) / (std + self.eps) + self.bias

## Feed Forward Network
It is a fully connected layer used both in encoder and decoder

$ FFN(x) = max(0, xW_1 + b_1)W_2 + b_2 $



In [ ]:
class FeedForward(nn.Module):
    """
    Feed-Forward Block — a small neural network applied independently to
    each token's vector after the attention step.

    Why we need it: attention lets tokens communicate with each other and
    gather information. The feed-forward block then lets each token process
    and transform that gathered information on its own — no communication
    between tokens here, just individual computation.

    Structure: two linear layers with a ReLU activation in between.
        linear_1: expands each vector from d_model → d_ff  (wider = more capacity)
        ReLU:     introduces non-linearity so the network can learn complex patterns
        linear_2: compresses back from d_ff → d_model  (back to the expected shape)

    Typical sizes: d_model=512, d_ff=2048 (4× expansion), so each token's
    vector goes 512 → 2048 → 512 through this block.
    """

    def __init__(self, d_model: int, d_ff: int, dropout: float) -> None:
        super().__init__()

        # First linear layer: expands each token vector from d_model to d_ff.
        # nn.Linear(in, out) creates a learnable weight matrix of shape
        # (in, out) and a bias vector of shape (out,). During forward it
        # computes: output = input × weights + bias
        # The larger d_ff gives the network more intermediate neurons to
        # work with — more capacity to detect different patterns in the data.
        self.linear_1 = nn.Linear(d_model, d_ff)

        # Dropout randomly zeroes some neuron outputs during training with
        # probability `dropout` (e.g. 0.1 = 10% of values become 0).
        # This prevents the network from relying too heavily on any single
        # neuron, which helps it generalize to unseen data.
        # At inference time (model.eval()) dropout is automatically disabled.
        self.dropout = nn.Dropout(dropout)

        # Second linear layer: compresses back from d_ff to d_model.
        # This restores the original vector size so the output of this block
        # has the same shape as the input — required because the transformer
        # adds the input back (residual connection) right after this block.
        self.linear_2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        # x shape coming in: (batch_size, seq_len, d_model)
        # The same sequence of operations is applied to every token independently.

        # Step 1 — linear_1: project each token vector from d_model to d_ff.
        # Shape: (batch, seq_len, d_model) → (batch, seq_len, d_ff)
        # Each of the d_ff output values is a learned weighted sum of all
        # d_model input values — different neurons look for different patterns.
        x = self.linear_1(x)

        # Step 2 — ReLU: apply the activation function element-wise.
        # ReLU(x) = max(0, x): negative values become 0, positives pass through.
        # This is critical — without it, linear_1 followed by linear_2 would
        # just be one big linear transformation (linear × linear = linear),
        # no matter how large d_ff is. ReLU introduces non-linearity, letting
        # the network learn curved, complex relationships in the data.
        # Shape stays: (batch, seq_len, d_ff)
        x = torch.relu(x)

        # Step 3 — dropout: randomly zero some values during training.
        # Applied after ReLU so we drop activated neuron outputs.
        # Shape stays: (batch, seq_len, d_ff)
        x = self.dropout(x)

        # Step 4 — linear_2: project back from d_ff to d_model.
        # This compresses the expanded representation back to the original
        # size. The network has to summarize everything it computed in the
        # wider space into d_model values — forcing it to keep only what matters.
        # Shape: (batch, seq_len, d_ff) → (batch, seq_len, d_model)
        return self.linear_2(x)

        # NOTE: the original code writes all four steps in one line:
        #   return self.linear_2(self.dropout(torch.relu(self.linear_1(x))))
        # This is identical — functions are just nested inside each other.
        # The expanded version above is easier to read and debug.

## Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention — lets every token look at every other token
    and decide how much to attend to each one, from multiple perspectives.

    Why we need it: to understand a word you often need context from other
    words. "it" needs to find its antecedent, a verb needs to find its
    subject, etc. Attention computes these relationships explicitly.

    Why multiple heads: a single attention pass can only look for one kind
    of relationship at a time. Multiple heads run in parallel, each free to
    specialize in a different type of relationship (syntax, coreference,
    proximity, etc.), giving the model much richer understanding.
    """

    def __init__(self, d_model: int, h: int, dropout: float) -> None:
        super().__init__()

        # d_model: the size of each token's embedding vector (e.g. 512).
        self.d_model = d_model

        # h: number of attention heads running in parallel (e.g. 8).
        # Each head will work with a slice of d_model dimensions.
        self.h = h

        # Each head gets an equal share of the embedding dimensions.
        # d_model must divide evenly by h — if not, we can't split fairly.
        # The assert will crash with a clear message if this is violated,
        # catching a misconfiguration early rather than getting a cryptic
        # shape error later.
        assert d_model % h == 0, "d_model is not divisible by h"

        # d_k: how many dimensions each head works with.
        # Example: d_model=512, h=8 → d_k=64
        # Each head sees a 64-dimensional slice instead of the full 512.
        self.d_k = d_model // h

        # Three separate linear projections — one each for Query, Key, Value.
        # Even though q, k, v might be the same input (in self-attention),
        # these learned weight matrices project them into three different
        # spaces, letting the model learn different representations for
        # "what am I looking for" (Q), "what do I offer" (K), and
        # "what information do I carry" (V).
        # Shape of each: (d_model, d_model) — input and output size same.
        self.w_q = nn.Linear(d_model, d_model)  # projects input into query space
        self.w_k = nn.Linear(d_model, d_model)  # projects input into key space
        self.w_v = nn.Linear(d_model, d_model)  # projects input into value space

        # Final output projection: after all heads are concatenated back
        # together, w_o mixes and blends their outputs into one coherent
        # vector. This lets the model learn how to combine what different
        # heads found.
        self.w_o = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    @staticmethod
    def attention(query, key, value, mask, dropout: nn.Dropout):
        """
        The core attention computation — static because it's pure math
        that doesn't need access to any learned weights (those were already
        applied before this method is called).

        @staticmethod means you can call this as MultiHeadAttention.attention(...)
        without needing an instance of the class. It's just a function that
        lives inside the class for organizational clarity.
        """

        # d_k: size of each head's query/key vectors.
        # We read it from the last dimension of query rather than self.d_k
        # so this method stays self-contained and reusable.
        d_k = query.shape[-1]

        # --- Step 1: compute raw attention scores ---
        # query shape:  (batch, h, seq_len, d_k)
        # key shape:    (batch, h, seq_len, d_k)
        #
        # key.transpose(-2, -1) flips the last two dimensions:
        #   (batch, h, seq_len, d_k) → (batch, h, d_k, seq_len)
        #
        # query @ key.transpose gives: (batch, h, seq_len, seq_len)
        # Entry [b, head, i, j] = dot product of token i's query
        # with token j's key = "how much should token i attend to token j?"
        #
        # Dividing by sqrt(d_k) prevents scores from getting too large,
        # which would cause softmax to become too "peaky" (one token gets
        # almost all attention, everything else is ignored).
        attention_scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

        # --- Step 2: apply mask (optional) ---
        # The mask is a matrix of 1s (attend) and 0s (block).
        # Wherever mask == 0, we set the score to a huge negative number.
        # After softmax, e^(-1e9) ≈ 0, so those positions get ~zero weight
        # and are effectively invisible to the attending token.
        # Used for: hiding padding tokens, or hiding future tokens in decoder.
        if mask is not None:
            attention_scores.masked_fill_(mask == 0, -1e9)

        # --- Step 3: softmax — convert scores to probabilities ---
        # dim=-1 means softmax is applied across the last dimension (the keys),
        # so for each query token, its scores across all key tokens sum to 1.
        # These are the final attention weights: how much to attend to each token.
        attention_scores = attention_scores.softmax(dim=-1)
        # shape: (batch, h, seq_len, seq_len)

        # Optionally zero out some attention weights during training.
        # This encourages the model not to over-rely on any single token.
        if dropout is not None:
            attention_scores = dropout(attention_scores)

        # --- Step 4: collect values weighted by attention ---
        # attention_scores: (batch, h, seq_len, seq_len)
        # value:            (batch, h, seq_len, d_k)
        # result:           (batch, h, seq_len, d_k)
        #
        # For each token, this computes a weighted average of all value vectors,
        # where the weights come from attention_scores. Tokens with high scores
        # contribute more to the output. This is the "collected answer" —
        # information gathered from the most relevant other tokens.
        #
        # We also return attention_scores so they can be inspected/visualized
        # outside the model (useful for debugging and understanding what
        # the model learned to attend to).
        return (attention_scores @ value), attention_scores

    def forward(self, q, k, v, mask):
        # q, k, v shape: (batch_size, seq_len, d_model)
        # In self-attention (encoder), all three are the same input.
        # In cross-attention (decoder), q comes from the decoder and
        # k, v come from the encoder output.

        # --- Step 1: linear projections ---
        # Project q, k, v into their respective learned spaces.
        # Even if q == k == v (self-attention), these produce three
        # different tensors because w_q, w_k, w_v have different weights.
        # Shape stays: (batch, seq_len, d_model)
        query = self.w_q(q)
        key   = self.w_k(k)
        value = self.w_v(v)

        # --- Step 2: split into h heads ---
        # .view() reshapes the last dimension d_model into (h, d_k):
        #   (batch, seq_len, d_model) → (batch, seq_len, h, d_k)
        # .transpose(1, 2) swaps seq_len and h dimensions:
        #   (batch, seq_len, h, d_k) → (batch, h, seq_len, d_k)
        #
        # After this, dim 1 is the head index — PyTorch will treat each
        # head like an independent item in the batch, running all h heads
        # in parallel without any explicit loop.
        query = query.view(query.shape[0], query.shape[1], self.h, self.d_k).transpose(1, 2)
        key   = key.view(key.shape[0],     key.shape[1],   self.h, self.d_k).transpose(1, 2)
        value = value.view(value.shape[0], value.shape[1], self.h, self.d_k).transpose(1, 2)

        # --- Step 3: compute attention for all heads in parallel ---
        # x shape:                    (batch, h, seq_len, d_k)
        # self.attention_scores shape: (batch, h, seq_len, seq_len)
        # We save attention_scores as an attribute so they can be inspected
        # after the forward pass (useful for visualizing what the model attends to).
        x, self.attention_scores = MultiHeadAttention.attention(
            query, key, value, mask, self.dropout
        )

        # --- Step 4: reassemble all heads back into one tensor ---
        # .transpose(1, 2) swaps h and seq_len back:
        #   (batch, h, seq_len, d_k) → (batch, seq_len, h, d_k)
        # .contiguous() ensures the tensor's memory is laid out sequentially
        #   (transpose doesn't move data, just changes how it's indexed —
        #    .view() below requires the data to actually be contiguous in memory)
        # .view(..., self.h * self.d_k) merges the h and d_k dimensions:
        #   (batch, seq_len, h, d_k) → (batch, seq_len, d_model)
        #   where d_model = h * d_k  (just undoing the split from step 2)
        x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.h * self.d_k)

        # --- Step 5: final output projection ---
        # w_o mixes the concatenated head outputs together.
        # Each head found different things — w_o learns how to combine
        # those different perspectives into one coherent output vector.
        # Shape stays: (batch, seq_len, d_model)
        return self.w_o(x)